In [1]:
import sys, platform, subprocess
import torch

print("===== Environment Info =====")
print("OS:", platform.platform())
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA (torch built with):", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU compute capability:", torch.cuda.get_device_capability(0))
    print("GPU total memory (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

try:
    import transformers
    print("transformers:", transformers.__version__)
except ImportError:
    print("transformers: not installed")

result = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
).stdout

lines = result.splitlines()

# Drop the timestamp on the first line
if lines:
    lines = lines[1:]

print("\n".join(lines))

===== Environment Info =====
OS: Linux-6.6.122+-x86_64-with-glibc2.39
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu128
CUDA (torch built with): 12.8
cuDNN version: 91900
GPU available: True
GPU name: Tesla T4
GPU compute capability: (7, 5)
GPU total memory (GB): 15.637086208
transformers: 5.16.1
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4            

In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
import os
os.environ['PATH'] += ':/opt/nvidia/nsight-compute/2025.1.1/host/target-linux-x64'
!nsys --version   # sanity check - if this errors, go look at the PATH/install issue then

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
NVIDIA Nsight Systems version 2025.1.1.0


# QKV Proj Kernel investigation - Perfetto UI

## GPU warm-up

In [2]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!python first_layer_estimation.py --dtype float32 --reversed_batch false --only_batch 1 --no_save

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
config.json: 100% 665/665 [00:00<00:00, 782kB/s]

model.safetensors: downloading bytes:  37% 204M/548M [00:01<00:01, 208MB/s, 16.3MB/s  ]
model.safetensors: reconstructing file:  12% 67.1M/548M [00:01<00:13, 36.4MB/s]
model.safetensors: downloading bytes:  51% 279M/548M [00:02<00:01, 223MB/s, 23.0MB/s  ]
model.safetensors: downloading bytes:  81% 444M/548M [00:02<00:00, 310MB/s, 36.6MB/s  ]
model.safetensors: reconstructing file:  61% 335M/548M [00:02<00:01, 154MB/s, 25.0MB/s  ]
model.safetensors: downloading bytes: 100% 474M/474M [00:05<00:00, 94.1MB/s, 41.8MB/s  ]
model.safetensors: reconstructing file: 100% 548M/548M [00:05<00:00, 109MB/s, 43.4MB/s  ]
Loading weights: 100% 148/148 [00:00<00:00, 7006.52it/s]
generation_config.json: 100% 1

## Ascending order

In [3]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!python first_layer_estimation.py --dtype float32 --reversed_batch false

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
Loading weights: 100% 148/148 [00:00<00:00, 4792.01it/s]
VAL_TOKENS=919,552 (max_batch=64, target_tokens=262,144)
Dtype: torch.float32
Batch sizes to sweep (ascending): [1, 2, 4, 8, 16, 32, 64]
Repeats per batch_size: 5

=== batch_size=1 (iters=256, tokens to process=262,144) ===
  [repeat 1/5]
    Tokens processed:                            262,144
    Total elapsed (sum of 256 calls):          1,755.123 ms
    BLOCK0 Forward (kernel, model forward only): 6.856±0.230 ms/call
    QKV_PROJ only (kernel, real/non-ncu):        0.910±0.051 ms/call
    Time per token:                              6.6953 us/token
    GPU (before run → after run):                49→52C, 855→1230MHz, 26.66→67.94W
  [repeat 2/5]
    Tokens processed:               

## Descending order

In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!python first_layer_estimation.py --dtype float32 --reversed_batch true

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
Loading weights: 100% 148/148 [00:00<00:00, 4493.88it/s]
VAL_TOKENS=919,552 (max_batch=64, target_tokens=262,144)
Dtype: torch.float32
Batch sizes to sweep (descending): [64, 32, 16, 8, 4, 2, 1]
Repeats per batch_size: 5

=== batch_size=64 (iters=4, tokens to process=262,144) ===
  [repeat 1/5]
    Tokens processed:                            262,144
    Total elapsed (sum of 4 calls):          1,915.282 ms
    BLOCK0 Forward (kernel, model forward only): 478.820±1.782 ms/call
    Time per token:                              7.3062 us/token
    GPU (before run → after run):                59→64C, 750→990MHz, 29.97→64.22W
  [repeat 2/5]
    Tokens processed:                            262,144
    Total elapsed (sum of 4 calls):          1,93

## Analysis

In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!python analyze_csv.py --csv output/ascending/fineweb_block0_nsight_evaluation.csv

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
Loaded 100 rows from output/ascending/fineweb_block0_nsight_evaluation.csv
Batch sizes found: [1, 2, 4, 8, 16, 32, 64]

Ranked fastest -> slowest, by total time to process the same amount of tokens (target_tokens):
  batch_size=32  | total time:  1,962.062± 137.003 ms (1.217x vs B=1) | per-call:    245.258± 17.125 ms/call (26.300x vs B=1, linear would be 32x, efficiency vs linear=0.822) | per-token: 7.4847 us/token | n_repeats=10
  batch_size=1   | total time:  2,387.285±2,232.747 ms (1.000x vs B=1) | per-call:      9.325±  8.722 ms/call (1.000x vs B=1, linear would be 1x, efficiency vs linear=1.000) | per-token: 9.1068 us/token | n_repeats=15
  batch_size=4   | total time:  2,415.702±2,554.324 ms (0.988x vs B=1) | per-call:     37.745± 39.911 ms/call (4.048x vs B=1, linear would be 4x, efficiency vs linear=1.012) | per-token: 9.2152 us/token | n_repeats=15
  batch_size=2   | total time:  2,44

In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!python analyze_csv.py --csv output/descending/fineweb_block0_nsight_evaluation.csv

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
Loaded 35 rows from output/descending/fineweb_block0_nsight_evaluation.csv
Batch sizes found: [1, 2, 4, 8, 16, 32, 64]

Ranked fastest -> slowest, by total time to process the same amount of tokens (target_tokens):
  batch_size=1   | total time:  1,889.093±  12.989 ms (1.000x vs B=1) | per-call:      7.379±  0.051 ms/call (1.000x vs B=1, linear would be 1x, efficiency vs linear=1.000) | per-token: 7.2063 us/token | n_repeats=5
  batch_size=4   | total time:  1,890.283±   8.812 ms (0.999x vs B=1) | per-call:     29.536±  0.138 ms/call (4.003x vs B=1, linear would be 4x, efficiency vs linear=1.001) | per-token: 7.2109 us/token | n_repeats=5
  batch_size=8   | total time:  1,918.045±   8.459 ms (0.985x vs B=1) | per-call:     59.939±  0.264 ms/call (8.123x vs B=1, linear would be 8x, efficiency vs linear=1.015) | per-token: 7.3168 us/token | n_repeats=5
  batch_size=16  | total time:  1,931.390± 

# ncu — is the difference actually coming from the kernel itself?

The ascending/descending sweep above doesn't scale batch_size → forward time
purely linearly — some batch_sizes take relatively more or less time per sample
than a straight multiple of `batch_size=1` would predict. To see whether that's
really happening *inside* the GPU kernel (not just measurement noise or CPU-side
overhead), we drop down one level and profile one specific kernel launch per
`batch_size` with Nsight Compute (`ncu`).

**Why `QKV_PROJ`:** `model.py` wraps every sub-step of a block's forward
(`QKV_PROJ`, `ATTN_CORE`, `OUT_PROJ`, `MLP_FC`, `MLP_GELU`, `MLP_PROJ`, ...) in
its own NVTX range, so any of them could be profiled. `QKV_PROJ` (`self.c_attn`,
block 0) is picked because it's the very first real matmul in the block — the
same first weight matrix GPTQ's own per-layer calibration touches — so it's the
earliest, least-confounded single kernel inside the exact scope this whole
script already isolates (block 0, via the `StopIteration` hook).

**Why the `--launch-skip` value differs per `batch_size`:** `--nvtx-include
"MODEL_FORWARD_BLOCK0/BLOCK_0/ATTN/QKV_PROJ/"` scopes profiling to just the
`QKV_PROJ` calls made *inside* the timed loop (the outer `MODEL_FORWARD_BLOCK0`
range is only pushed there, not during warmup) — one call per `idx`, so
`iters_for_B` matching launches total for that `batch_size` (printed by the
script itself as `iters=...`, derived as `target_tokens // (batch_size *
token_size)`). `--launch-skip <iters_for_B - 1> --launch-count 1` skips all but
the very last of those, profiling only the final, fully warmed-up, steady-state
call — the same "last repeat" convention already used elsewhere in this script
(e.g. the Perfetto trace export). Since `iters_for_B` is a different number for
every `batch_size` (256, 128, 64, 32, 16, 4 for B=1,2,4,8,16,64 — fewer, bigger
calls needed to reach the same `target_tokens`), the skip value that lands on
"the last one" is different for each: `255, 127, 63, 31, 15, 3` respectively.

**Why `--kernel-name regex:.*sgemm.*`:** one `QKV_PROJ`-scoped call can still
launch more than one actual CUDA kernel (the GEMM itself, plus incidental
bookkeeping); this filters down to the GEMM (`sgemm`, single-precision matrix
multiply) specifically, which is the compute this investigation actually cares
about.

**Why `--set full`:** collects the complete metric set so we can tell *what
kind* of bottleneck this kernel has at each `batch_size` — compute-bound (SM
busy), memory-bound (L1/L2/DRAM busy), or both ("co-bound") — and how that
changes as `batch_size` grows.

**Why `--skip_perfetto`:** `ncu` and `torch.profiler` both want exclusive CUPTI
access; running both at once raises
`CUPTI_ERROR_MULTIPLE_SUBSCRIBERS_NOT_SUPPORTED`.

In [ ]:
# B=1 set full
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!mkdir -p output/compute_bound
!ncu --nvtx --nvtx-include "MODEL_FORWARD_BLOCK0/BLOCK_0/ATTN/QKV_PROJ/" \
    --kernel-name regex:.*sgemm.* \
    --launch-skip 255 --launch-count 1 \
    --set full \
    -o output/compute_bound/qkv_proj_b1_full \
    python first_layer_estimation.py --dtype float32 --reversed_batch false --only_batch 1 --skip_perfetto --repeats 1 --skip_gpu_status --no_save

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
Loading weights: 100% 148/148 [00:00<00:00, 3566.13it/s]
==PROF== Connected to process 3190 (/usr/bin/python3.13)
VAL_TOKENS=274,432 (max_batch=1, target_tokens=262,144)
Dtype: torch.float32
Batch sizes to sweep (ascending): [1]
Repeats per batch_size: 1

=== batch_size=1 (iters=256, tokens to process=262,144) ===
==PROF== Profiling "volta_sgemm_128x64_tn": 0%....50%....100% - 31 passes
  [repeat 1/1]
    Tokens processed:                            262,144
    Total elapsed (sum of 256 calls):          10,211.033 ms
    BLOCK0 Forward (kernel, model forward only): 39.887±471.451 ms/call
    Time per token:                              38.9520 us/token

[batch_size=1] over 1 repeats:
    Tokens processed:                            262,144


In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!ncu --import output/compute_bound/qkv_proj_b1_full.ncu-rep \
    --metrics sm__throughput.avg.pct_of_peak_sustained_elapsed,gpu__compute_memory_throughput.avg.pct_of_peak_sustained_elapsed

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
[3190] python3.13@127.0.0.1
  volta_sgemm_128x64_tn (18, 16, 2)x(128, 1, 1), Context 1, Stream 7, Device 0, CC 7.5

    NVTX Push/Pop Stack for Thread 3190:
      <default domain>
        <0,MODEL_FORWARD_BLOCK0>
        <1,BLOCK_0>
        <2,ATTN>
        <3,QKV_PROJ>
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    Memory Throughput                 %        44.48
    Compute (SM) Throughput           %        84.45
    ----------------------- ----------- ------------

    INF   This workload is utilizing greater than 80.0% of the available compute or memory performance of the device.   
          To further improve performance, work will likely need to be shifted from the most utilized to another unit.   
          Start by analyzing wo

In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!ncu --import output/compute_bound/qkv_proj_b1_full.ncu-rep \
    --page raw \
    --metrics sm__cycles_elapsed.avg.per_second

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
[3190] python3.13@127.0.0.1
  volta_sgemm_128x64_tn (18, 16, 2)x(128, 1, 1), Context 1, Stream 7, Device 0, CC 7.5

    NVTX Push/Pop Stack for Thread 3190:
      <default domain>
        <0,MODEL_FORWARD_BLOCK0>
        <1,BLOCK_0>
        <2,ATTN>
        <3,QKV_PROJ>
  Metric Name                       Metric Unit         Metric Value
  --------------------------------- ----------- --------------------
  sm__cycles_elapsed.avg.per_second         Mhz               583.56



In [ ]:
# B=2 set full
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!mkdir -p output/compute_bound
!ncu --nvtx --nvtx-include "MODEL_FORWARD_BLOCK0/BLOCK_0/ATTN/QKV_PROJ/" \
    --kernel-name regex:.*sgemm.* \
    --launch-skip 127 --launch-count 1 \
    --set full \
    -o output/compute_bound/qkv_proj_b2_full \
    python first_layer_estimation.py --dtype float32 --reversed_batch false --only_batch 2 --skip_perfetto --repeats 1 --skip_gpu_status --no_save

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
Loading weights: 100% 148/148 [00:00<00:00, 18530.06it/s]
==PROF== Connected to process 3451 (/usr/bin/python3.13)
VAL_TOKENS=284,672 (max_batch=2, target_tokens=262,144)
Dtype: torch.float32
Batch sizes to sweep (ascending): [2]
Repeats per batch_size: 1

=== batch_size=2 (iters=128, tokens to process=262,144) ===
==PROF== Profiling "volta_sgemm_128x64_tn": 0%....50%....100% - 31 passes
  [repeat 1/1]
    Tokens processed:                            262,144
    Total elapsed (sum of 128 calls):          10,260.806 ms
    BLOCK0 Forward (kernel, model forward only): 80.163±736.462 ms/call
    Time per token:                              39.1419 us/token

[batch_size=2] over 1 repeats:
    Tokens processed:                            262,144

In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!ncu --import output/compute_bound/qkv_proj_b2_full.ncu-rep \
    --metrics sm__throughput.avg.pct_of_peak_sustained_elapsed,gpu__compute_memory_throughput.avg.pct_of_peak_sustained_elapsed

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
[3451] python3.13@127.0.0.1
  volta_sgemm_128x64_tn (18, 32, 2)x(128, 1, 1), Context 1, Stream 7, Device 0, CC 7.5

    NVTX Push/Pop Stack for Thread 3451:
      <default domain>
        <0,MODEL_FORWARD_BLOCK0>
        <1,BLOCK_0>
        <2,ATTN>
        <3,QKV_PROJ>
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    Memory Throughput                 %        46.17
    Compute (SM) Throughput           %        87.66
    ----------------------- ----------- ------------

    INF   This workload is utilizing greater than 80.0% of the available compute or memory performance of the device.   
          To further improve performance, work will likely need to be shifted from the most utilized to another unit.   
          Start by analyzing wo

In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!ncu --import output/compute_bound/qkv_proj_b2_full.ncu-rep \
    --page raw \
    --metrics sm__cycles_elapsed.avg.per_second

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
[3451] python3.13@127.0.0.1
  volta_sgemm_128x64_tn (18, 32, 2)x(128, 1, 1), Context 1, Stream 7, Device 0, CC 7.5

    NVTX Push/Pop Stack for Thread 3451:
      <default domain>
        <0,MODEL_FORWARD_BLOCK0>
        <1,BLOCK_0>
        <2,ATTN>
        <3,QKV_PROJ>
  Metric Name                       Metric Unit         Metric Value
  --------------------------------- ----------- --------------------
  sm__cycles_elapsed.avg.per_second         Mhz               585.20



In [ ]:
# B=4 set full
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!mkdir -p output/compute_bound
!ncu --nvtx --nvtx-include "MODEL_FORWARD_BLOCK0/BLOCK_0/ATTN/QKV_PROJ/" \
    --kernel-name regex:.*sgemm.* \
    --launch-skip 63 --launch-count 1 \
    --set full \
    -o output/compute_bound/qkv_proj_b4_full \
    python first_layer_estimation.py --dtype float32 --reversed_batch false --only_batch 4 --skip_perfetto --repeats 1 --skip_gpu_status --no_save

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
Loading weights: 100% 148/148 [00:00<00:00, 18335.21it/s]
==PROF== Connected to process 3675 (/usr/bin/python3.13)
VAL_TOKENS=305,152 (max_batch=4, target_tokens=262,144)
Dtype: torch.float32
Batch sizes to sweep (ascending): [4]
Repeats per batch_size: 1

=== batch_size=4 (iters=64, tokens to process=262,144) ===
==PROF== Profiling "volta_sgemm_128x64_tn": 0%....50%....100% - 31 passes
  [repeat 1/1]
    Tokens processed:                            262,144
    Total elapsed (sum of 64 calls):          11,872.505 ms
    BLOCK0 Forward (kernel, model forward only): 185.508±1,258.886 ms/call
    Time per token:                              45.2900 us/token

[batch_size=4] over 1 repeats:
    Tokens processed:                            262,14

In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!ncu --import output/compute_bound/qkv_proj_b4_full.ncu-rep \
    --metrics sm__throughput.avg.pct_of_peak_sustained_elapsed,gpu__compute_memory_throughput.avg.pct_of_peak_sustained_elapsed

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
[3675] python3.13@127.0.0.1
  volta_sgemm_128x64_tn (18, 64, 1)x(128, 1, 1), Context 1, Stream 7, Device 0, CC 7.5

    NVTX Push/Pop Stack for Thread 3675:
      <default domain>
        <0,MODEL_FORWARD_BLOCK0>
        <1,BLOCK_0>
        <2,ATTN>
        <3,QKV_PROJ>
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    Memory Throughput                 %        46.08
    Compute (SM) Throughput           %        90.52
    ----------------------- ----------- ------------

    INF   This workload is utilizing greater than 80.0% of the available compute or memory performance of the device.   
          To further improve performance, work will likely need to be shifted from the most utilized to another unit.   
          Start by analyzing wo

In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!ncu --import output/compute_bound/qkv_proj_b4_full.ncu-rep \
    --page raw \
    --metrics sm__cycles_elapsed.avg.per_second

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
[3675] python3.13@127.0.0.1
  volta_sgemm_128x64_tn (18, 64, 1)x(128, 1, 1), Context 1, Stream 7, Device 0, CC 7.5

    NVTX Push/Pop Stack for Thread 3675:
      <default domain>
        <0,MODEL_FORWARD_BLOCK0>
        <1,BLOCK_0>
        <2,ATTN>
        <3,QKV_PROJ>
  Metric Name                       Metric Unit         Metric Value
  --------------------------------- ----------- --------------------
  sm__cycles_elapsed.avg.per_second         Mhz               585.19



In [ ]:
# B=8 set full
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!mkdir -p output/compute_bound
!ncu --nvtx --nvtx-include "MODEL_FORWARD_BLOCK0/BLOCK_0/ATTN/QKV_PROJ/" \
    --kernel-name regex:.*sgemm.* \
    --launch-skip 31 --launch-count 1 \
    --set full \
    -o output/compute_bound/qkv_proj_b8_full \
    python first_layer_estimation.py --dtype float32 --reversed_batch false --only_batch 8 --skip_perfetto --repeats 1 --skip_gpu_status --no_save

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
==ERROR== File qkv_proj_b8_full.ncu-rep already exists. Use '-f' for overwriting the file.


In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!ncu --import output/compute_bound/qkv_proj_b8_full.ncu-rep \
    --metrics sm__throughput.avg.pct_of_peak_sustained_elapsed,gpu__compute_memory_throughput.avg.pct_of_peak_sustained_elapsed

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
[2226] python3.13@127.0.0.1
  volta_sgemm_128x128_tn (18, 64, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5

    NVTX Push/Pop Stack for Thread 2226:
      <default domain>
        <0,MODEL_FORWARD_BLOCK0>
        <1,BLOCK_0>
        <2,ATTN>
        <3,QKV_PROJ>
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    Memory Throughput                 %        40.21
    Compute (SM) Throughput           %        94.58
    ----------------------- ----------- ------------

    INF   This workload is utilizing greater than 80.0% of the available compute or memory performance of the device.   
          To further improve performance, work will likely need to be shifted from the most utilized to another unit.   
          Start by analyzing w

In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!ncu --import output/compute_bound/qkv_proj_b8_full.ncu-rep \
    --page raw \
    --metrics sm__cycles_elapsed.avg.per_second

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
[2226] python3.13@127.0.0.1
  volta_sgemm_128x128_tn (18, 64, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5

    NVTX Push/Pop Stack for Thread 2226:
      <default domain>
        <0,MODEL_FORWARD_BLOCK0>
        <1,BLOCK_0>
        <2,ATTN>
        <3,QKV_PROJ>
  Metric Name                       Metric Unit         Metric Value
  --------------------------------- ----------- --------------------
  sm__cycles_elapsed.avg.per_second         Mhz               585.44



In [ ]:
# B=16
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!mkdir -p output/compute_bound
!ncu --nvtx --nvtx-include "MODEL_FORWARD_BLOCK0/BLOCK_0/ATTN/QKV_PROJ/" \
    --kernel-name regex:.*sgemm.* \
    --launch-skip 15 --launch-count 1 \
    --set full \
    -o output/compute_bound/qkv_proj_b16_full \
    python first_layer_estimation.py --dtype float32 --reversed_batch false --only_batch 16 --skip_perfetto --repeats 1 --skip_gpu_status --no_save

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
==ERROR== File qkv_proj_b16_full.ncu-rep already exists. Use '-f' for overwriting the file.


In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!ncu --import output/compute_bound/qkv_proj_b16_full.ncu-rep \
    --metrics sm__throughput.avg.pct_of_peak_sustained_elapsed,gpu__compute_memory_throughput.avg.pct_of_peak_sustained_elapsed

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
[2451] python3.13@127.0.0.1
  volta_sgemm_128x128_tn (18, 128, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5

    NVTX Push/Pop Stack for Thread 2451:
      <default domain>
        <0,MODEL_FORWARD_BLOCK0>
        <1,BLOCK_0>
        <2,ATTN>
        <3,QKV_PROJ>
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    Memory Throughput                 %        40.29
    Compute (SM) Throughput           %        94.73
    ----------------------- ----------- ------------

    INF   This workload is utilizing greater than 80.0% of the available compute or memory performance of the device.   
          To further improve performance, work will likely need to be shifted from the most utilized to another unit.   
          Start by analyzing 

In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!ncu --import output/compute_bound/qkv_proj_b16_full.ncu-rep \
    --page raw \
    --metrics sm__cycles_elapsed.avg.per_second

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
[2451] python3.13@127.0.0.1
  volta_sgemm_128x128_tn (18, 128, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5

    NVTX Push/Pop Stack for Thread 2451:
      <default domain>
        <0,MODEL_FORWARD_BLOCK0>
        <1,BLOCK_0>
        <2,ATTN>
        <3,QKV_PROJ>
  Metric Name                       Metric Unit         Metric Value
  --------------------------------- ----------- --------------------
  sm__cycles_elapsed.avg.per_second         Mhz               585.05



In [ ]:
# B=32 set full
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!mkdir -p output/compute_bound
!ncu --nvtx --nvtx-include "MODEL_FORWARD_BLOCK0/BLOCK_0/ATTN/QKV_PROJ/" \
    --kernel-name regex:.*sgemm.* \
    --launch-skip 7 --launch-count 1 \
    --set full \
    -o output/compute_bound/qkv_proj_b32_full \
    python first_layer_estimation.py --dtype float32 --reversed_batch false --only_batch 32 --repeats 1 --skip_perfetto --skip_gpu_status --no_save

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
Loading weights: 100% 148/148 [00:00<00:00, 5064.80it/s]
==PROF== Connected to process 4017 (/usr/bin/python3.13)
VAL_TOKENS=591,872 (max_batch=32, target_tokens=262,144)
Dtype: torch.float32
Batch sizes to sweep (ascending): [32]
Repeats per batch_size: 1

=== batch_size=32 (iters=8, tokens to process=262,144) ===
==PROF== Profiling "volta_sgemm_128x128_tn": 0%.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://docs.nvidia.com/nsight-compute/ProfilingGuide/index.html#replay for more details.
...50%....100% - 31 passes
  [repeat 1/1]
    Tokens processed:                     

In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!ncu --import output/compute_bound/qkv_proj_b32_full.ncu-rep \
    --metrics sm__throughput.avg.pct_of_peak_sustained_elapsed,gpu__compute_memory_throughput.avg.pct_of_peak_sustained_elapsed

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
[4017] python3.13@127.0.0.1
  volta_sgemm_128x128_tn (18, 256, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5

    NVTX Push/Pop Stack for Thread 4017:
      <default domain>
        <0,MODEL_FORWARD_BLOCK0>
        <1,BLOCK_0>
        <2,ATTN>
        <3,QKV_PROJ>
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    Memory Throughput                 %        40.49
    Compute (SM) Throughput           %        95.21
    ----------------------- ----------- ------------

    INF   This workload is utilizing greater than 80.0% of the available compute or memory performance of the device.   
          To further improve performance, work will likely need to be shifted from the most utilized to another unit.   
          Start by analyzing 

In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!ncu --import output/compute_bound/qkv_proj_b32_full.ncu-rep \
    --page raw \
    --metrics sm__cycles_elapsed.avg.per_second

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
[4017] python3.13@127.0.0.1
  volta_sgemm_128x128_tn (18, 256, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5

    NVTX Push/Pop Stack for Thread 4017:
      <default domain>
        <0,MODEL_FORWARD_BLOCK0>
        <1,BLOCK_0>
        <2,ATTN>
        <3,QKV_PROJ>
  Metric Name                       Metric Unit         Metric Value
  --------------------------------- ----------- --------------------
  sm__cycles_elapsed.avg.per_second         Mhz               584.92



In [ ]:
# B=64
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!mkdir -p output/compute_bound
!ncu --nvtx --nvtx-include "MODEL_FORWARD_BLOCK0/BLOCK_0/ATTN/QKV_PROJ/" \
    --kernel-name regex:.*sgemm.* \
    --launch-skip 3 --launch-count 1 \
    --set full \
    -o output/compute_bound/qkv_proj_b64_full \
    python first_layer_estimation.py --dtype float32 --reversed_batch false --only_batch 64 --skip_perfetto --repeats 1 --skip_gpu_status --no_save

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
Loading weights: 100% 148/148 [00:00<00:00, 16559.70it/s]
==PROF== Connected to process 4334 (/usr/bin/python3.13)
VAL_TOKENS=919,552 (max_batch=64, target_tokens=262,144)
Dtype: torch.float32
Batch sizes to sweep (ascending): [64]
Repeats per batch_size: 1

=== batch_size=64 (iters=4, tokens to process=262,144) ===
==PROF== Profiling "volta_sgemm_128x128_tn": 0%
==WARNING== Backing up device memory in system memory. Kernel replay might be slow. Consider using "--replay-mode application" to avoid memory save-and-restore.
.
==WARNING== Launching the workload is taking more time than expected. If this continues to hang, terminate the profile and re-try by profiling the range of all related launches using '--replay-mode range'. See https://doc

In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!ncu --import output/compute_bound/qkv_proj_b64_full.ncu-rep \
    --metrics sm__throughput.avg.pct_of_peak_sustained_elapsed,gpu__compute_memory_throughput.avg.pct_of_peak_sustained_elapsed

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
[4334] python3.13@127.0.0.1
  volta_sgemm_128x128_tn (18, 512, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5

    NVTX Push/Pop Stack for Thread 4334:
      <default domain>
        <0,MODEL_FORWARD_BLOCK0>
        <1,BLOCK_0>
        <2,ATTN>
        <3,QKV_PROJ>
    Section: GPU Speed Of Light Throughput
    ----------------------- ----------- ------------
    Metric Name             Metric Unit Metric Value
    ----------------------- ----------- ------------
    Memory Throughput                 %        40.67
    Compute (SM) Throughput           %        95.62
    ----------------------- ----------- ------------

    INF   This workload is utilizing greater than 80.0% of the available compute or memory performance of the device.   
          To further improve performance, work will likely need to be shifted from the most utilized to another unit.   
          Start by analyzing 

In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!ncu --import output/compute_bound/qkv_proj_b64_full.ncu-rep \
    --page raw \
    --metrics sm__cycles_elapsed.avg.per_second

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
[4334] python3.13@127.0.0.1
  volta_sgemm_128x128_tn (18, 512, 1)x(256, 1, 1), Context 1, Stream 7, Device 0, CC 7.5

    NVTX Push/Pop Stack for Thread 4334:
      <default domain>
        <0,MODEL_FORWARD_BLOCK0>
        <1,BLOCK_0>
        <2,ATTN>
        <3,QKV_PROJ>
  Metric Name                       Metric Unit         Metric Value
  --------------------------------- ----------- --------------------
  sm__cycles_elapsed.avg.per_second         Mhz               585.10



**Finding:** Compute (SM) Throughput and L1/TEX (memory) Throughput are both high
across every `batch_size` here — this kernel is co-bound, not purely one or the
other — and Compute% *plateaus* after `batch_size=8` (~94.6% → ~95.6%, basically
flat from there on). That plateau explains the *diminishing* returns at larger
batch_sizes: the SM is already close to fully busy by B=8, so packing in more
samples per call stops buying much more throughput.

It doesn't explain the sudden, erratic *slowdowns* seen in some large-batch runs
(worse toward the end of a long sweep) — that's what the power check below is
for.

## ncu - Pure kernel time estimation
ncu fix the clock to 585 MHz

In [ ]:
# GPU warm up
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!python first_layer_estimation.py --dtype float32 --reversed_batch false --only_batch 1 --no_save

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
config.json: 100% 665/665 [00:00<00:00, 756kB/s]

model.safetensors: downloading bytes:  17% 95.2M/548M [00:00<00:02, 172MB/s, 6.68MB/s  ]
model.safetensors: downloading bytes:  37% 205M/548M [00:01<00:01, 239MB/s, 16.7MB/s  ]
model.safetensors: downloading bytes:  67% 367M/548M [00:01<00:00, 223MB/s, 32.4MB/s  ]
model.safetensors: downloading bytes:  86% 472M/548M [00:02<00:00, 289MB/s, 38.9MB/s  ]
model.safetensors: reconstructing file:  61% 335M/548M [00:04<00:03, 67.8MB/s, 24.7MB/s  ]
model.safetensors: downloading bytes: 100% 474M/474M [00:08<00:00, 57.6MB/s, 42.0MB/s  ]
model.safetensors: reconstructing file: 100% 548M/548M [00:08<00:00, 66.5MB/s, 36.4MB/s  ]
Loading weights: 100% 148/148 [00:00<00:00, 7220.12it/s]
generation_config.j

In [ ]:
%%bash
mkdir -p output/compute_bound

for B in 1 2 4 8 16 32 64; do
    skip=$(( 256 / B - 1 ))   # existing launch-skip formula, based on target_tokens=262144, T=1024
    for r in 1 2 3 4 5; do
        ncu --nvtx --nvtx-include "MODEL_FORWARD_BLOCK0/BLOCK_0/ATTN/QKV_PROJ/" \
            --kernel-name regex:.*sgemm.* \
            --launch-skip $skip --launch-count 1 \
            --metrics gpu__time_duration.sum \
            -o output/compute_bound/qkv_proj_b${B}_dur_r${r} \
            python first_layer_estimation.py --dtype float32 --reversed_batch false --only_batch $B --skip_perfetto --repeats 1 --skip_gpu_status --no_save
    done
done

loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
==PROF== Connected to process 1595 (/usr/bin/python3.13)
==PROF== Profiling "volta_sgemm_128x64_tn": 0%....50%....100% - 1 pass
VAL_TOKENS=274,432 (max_batch=1, target_tokens=262,144)
Dtype: torch.float32
Batch sizes to sweep (ascending): [1]
Repeats per batch_size: 1

=== batch_size=1 (iters=256, tokens to process=262,144) ===
  [repeat 1/1]
    Tokens processed:                            262,144
    Total elapsed (sum of 256 calls):          2,960.664 ms
    BLOCK0 Forward (kernel, model forward only): 11.565±47.139 ms/call
    QKV_PROJ only (kernel, real/non-ncu):        4.399±46.957 ms/call
    Time per token:                              11.2940 us/token

[batch_size=1] over 1 repeats:
    Tokens processed:                            262,144
    Total elapsed:                                2,960.664±0.000 ms
    BLOCK0 Forwar

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4705.73it/s]


In [ ]:
%%bash
for B in 1 2 4 8 16 32 64; do
    for r in 1 2 3 4 5; do
        ncu --import output/compute_bound/qkv_proj_b${B}_dur_r${r}.ncu-rep \
            --csv --page raw --metrics gpu__time_duration.sum
    done
done > output/compute_bound/duration_raw.csv

# Power check — why do the timings spike like that?

The `ncu` drill-down above explains diminishing returns, but not the erratic
slowdowns actually observed for some large-batch runs, especially later in a
long sweep. To check whether that's a GPU power/thermal effect rather than
something about the kernel itself, this logs `nvidia-smi`'s
`power.draw` / `power.limit` / `clocks.sm` / `clocks_throttle_reasons.sw_power_cap`
at 200ms resolution while a full sweep runs underneath it.

**Finding:** `sw_power_cap` (software power-cap throttling) shows Active
specifically in the batch=32/64 region, and gets *more* persistent the later/
larger the batch — i.e. sustained high compute utilization eventually pushes
the GPU against its power limit, and the driver clocks it down to stay under
that cap. That power-cap throttling, not something intrinsic to the kernel or
`batch_size` itself, is what's actually behind the erratic large-batch
slowdowns.

In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation


In [ ]:
%%bash
BATCH=1
echo "# Batch size = ${BATCH}" > throttle_log_b${BATCH}.csv
nvidia-smi --query-gpu=timestamp,power.draw,power.limit,clocks.sm,clocks_throttle_reasons.sw_power_cap \
    --format=csv -lms 200 >> throttle_log_b${BATCH}.csv &
LOGGER_PID=$!

python first_layer_estimation.py --dtype float32 --reversed_batch false --only_batch ${BATCH} --no_save

kill $LOGGER_PID

loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
VAL_TOKENS=274,432 (max_batch=1, target_tokens=262,144)
Dtype: torch.float32
Batch sizes to sweep (ascending): [1]
Repeats per batch_size: 5

=== batch_size=1 (iters=256, tokens to process=262,144) ===
  [repeat 1/5]
    Tokens processed:                            262,144
    Total elapsed (sum of 256 calls):          1,724.693 ms
    BLOCK0 Forward (kernel, model forward only): 6.737±0.831 ms/call
    Time per token:                              6.5792 us/token
    GPU (before run → after run):                39→42C, 585→1365MHz, 26.16→68.92W
  [repeat 2/5]
    Tokens processed:                            262,144
    Total elapsed (sum of 256 calls):          1,671.322 ms
    BLOCK0 Forward (kernel, model forward only): 6.529±0.081 ms/call
    Time per token:                              6.3756 us/token
    GPU (before run → after

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6055.87it/s]


In [ ]:
%%bash
BATCH=2
echo "# Batch size = ${BATCH}" > throttle_log_b${BATCH}.csv
nvidia-smi --query-gpu=timestamp,power.draw,power.limit,clocks.sm,clocks_throttle_reasons.sw_power_cap \
    --format=csv -lms 200 >> throttle_log_b${BATCH}.csv &
LOGGER_PID=$!

python first_layer_estimation.py --dtype float32 --reversed_batch false --only_batch ${BATCH} --no_save

kill $LOGGER_PID

loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
VAL_TOKENS=284,672 (max_batch=2, target_tokens=262,144)
Dtype: torch.float32
Batch sizes to sweep (ascending): [2]
Repeats per batch_size: 5

=== batch_size=2 (iters=128, tokens to process=262,144) ===
  [repeat 1/5]
    Tokens processed:                            262,144
    Total elapsed (sum of 128 calls):          1,702.659 ms
    BLOCK0 Forward (kernel, model forward only): 13.302±0.682 ms/call
    Time per token:                              6.4951 us/token
    GPU (before run → after run):                43→46C, 1005→1545MHz, 27.05→66.73W
  [repeat 2/5]
    Tokens processed:                            262,144
    Total elapsed (sum of 128 calls):          1,742.109 ms
    BLOCK0 Forward (kernel, model forward only): 13.610±2.241 ms/call
    Time per token:                              6.6456 us/token
    GPU (before run → af

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4636.08it/s]


In [ ]:
%%bash
BATCH=4
echo "# Batch size = ${BATCH}" > throttle_log_b${BATCH}.csv
nvidia-smi --query-gpu=timestamp,power.draw,power.limit,clocks.sm,clocks_throttle_reasons.sw_power_cap \
    --format=csv -lms 200 >> throttle_log_b${BATCH}.csv &
LOGGER_PID=$!

python first_layer_estimation.py --dtype float32 --reversed_batch false --only_batch ${BATCH} --no_save

kill $LOGGER_PID

loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
VAL_TOKENS=305,152 (max_batch=4, target_tokens=262,144)
Dtype: torch.float32
Batch sizes to sweep (ascending): [4]
Repeats per batch_size: 5

=== batch_size=4 (iters=64, tokens to process=262,144) ===
  [repeat 1/5]
    Tokens processed:                            262,144
    Total elapsed (sum of 64 calls):          1,654.736 ms
    BLOCK0 Forward (kernel, model forward only): 25.855±0.321 ms/call
    Time per token:                              6.3123 us/token
    GPU (before run → after run):                47→50C, 1005→1035MHz, 27.55→69.41W
  [repeat 2/5]
    Tokens processed:                            262,144
    Total elapsed (sum of 64 calls):          1,662.588 ms
    BLOCK0 Forward (kernel, model forward only): 25.978±0.236 ms/call
    Time per token:                              6.3423 us/token
    GPU (before run → after

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 5711.84it/s]


In [ ]:
%%bash
BATCH=8
echo "# Batch size = ${BATCH}" > throttle_log_b${BATCH}.csv
nvidia-smi --query-gpu=timestamp,power.draw,power.limit,clocks.sm,clocks_throttle_reasons.sw_power_cap \
    --format=csv -lms 200 >> throttle_log_b${BATCH}.csv &
LOGGER_PID=$!

python first_layer_estimation.py --dtype float32 --reversed_batch false --only_batch ${BATCH} --no_save

kill $LOGGER_PID

loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
VAL_TOKENS=346,112 (max_batch=8, target_tokens=262,144)
Dtype: torch.float32
Batch sizes to sweep (ascending): [8]
Repeats per batch_size: 5

=== batch_size=8 (iters=32, tokens to process=262,144) ===
  [repeat 1/5]
    Tokens processed:                            262,144
    Total elapsed (sum of 32 calls):          1,676.871 ms
    BLOCK0 Forward (kernel, model forward only): 52.402±0.561 ms/call
    Time per token:                              6.3968 us/token
    GPU (before run → after run):                49→52C, 1005→900MHz, 28.72→62.44W
  [repeat 2/5]
    Tokens processed:                            262,144
    Total elapsed (sum of 32 calls):          1,679.426 ms
    BLOCK0 Forward (kernel, model forward only): 52.482±0.743 ms/call
    Time per token:                              6.4065 us/token
    GPU (before run → after 

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4570.20it/s]


In [ ]:
%%bash
BATCH=16
echo "# Batch size = ${BATCH}" > throttle_log_b${BATCH}.csv
nvidia-smi --query-gpu=timestamp,power.draw,power.limit,clocks.sm,clocks_throttle_reasons.sw_power_cap \
    --format=csv -lms 200 >> throttle_log_b${BATCH}.csv &
LOGGER_PID=$!

python first_layer_estimation.py --dtype float32 --reversed_batch false --only_batch ${BATCH} --no_save

kill $LOGGER_PID

loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
VAL_TOKENS=428,032 (max_batch=16, target_tokens=262,144)
Dtype: torch.float32
Batch sizes to sweep (ascending): [16]
Repeats per batch_size: 5

=== batch_size=16 (iters=16, tokens to process=262,144) ===
  [repeat 1/5]
    Tokens processed:                            262,144
    Total elapsed (sum of 16 calls):          1,708.388 ms
    BLOCK0 Forward (kernel, model forward only): 106.774±0.975 ms/call
    Time per token:                              6.5170 us/token
    GPU (before run → after run):                51→55C, 975→1005MHz, 27.75→37.63W
  [repeat 2/5]
    Tokens processed:                            262,144
    Total elapsed (sum of 16 calls):          1,715.352 ms
    BLOCK0 Forward (kernel, model forward only): 107.209±0.738 ms/call
    Time per token:                              6.5435 us/token
    GPU (before run → a

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4753.99it/s]


In [ ]:
%%bash
BATCH=32
echo "# Batch size = ${BATCH}" > throttle_log_b${BATCH}.csv
nvidia-smi --query-gpu=timestamp,power.draw,power.limit,clocks.sm,clocks_throttle_reasons.sw_power_cap \
    --format=csv -lms 200 >> throttle_log_b${BATCH}.csv &
LOGGER_PID=$!

python first_layer_estimation.py --dtype float32 --reversed_batch false --only_batch ${BATCH} --no_save

kill $LOGGER_PID

loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
VAL_TOKENS=591,872 (max_batch=32, target_tokens=262,144)
Dtype: torch.float32
Batch sizes to sweep (ascending): [32]
Repeats per batch_size: 5

=== batch_size=32 (iters=8, tokens to process=262,144) ===
  [repeat 1/5]
    Tokens processed:                            262,144
    Total elapsed (sum of 8 calls):          1,747.915 ms
    BLOCK0 Forward (kernel, model forward only): 218.489±0.706 ms/call
    Time per token:                              6.6678 us/token
    GPU (before run → after run):                53→57C, 795→810MHz, 27.85→69.12W
  [repeat 2/5]
    Tokens processed:                            262,144
    Total elapsed (sum of 8 calls):          1,748.385 ms
    BLOCK0 Forward (kernel, model forward only): 218.548±2.582 ms/call
    Time per token:                              6.6696 us/token
    GPU (before run → after

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 4484.66it/s]


In [ ]:
%%bash
BATCH=64
echo "# Batch size = ${BATCH}" > throttle_log_b${BATCH}.csv
nvidia-smi --query-gpu=timestamp,power.draw,power.limit,clocks.sm,clocks_throttle_reasons.sw_power_cap \
    --format=csv -lms 200 >> throttle_log_b${BATCH}.csv &
LOGGER_PID=$!

python first_layer_estimation.py --dtype float32 --reversed_batch false --only_batch ${BATCH} --no_save

kill $LOGGER_PID

loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
VAL_TOKENS=919,552 (max_batch=64, target_tokens=262,144)
Dtype: torch.float32
Batch sizes to sweep (ascending): [64]
Repeats per batch_size: 5

=== batch_size=64 (iters=4, tokens to process=262,144) ===
  [repeat 1/5]
    Tokens processed:                            262,144
    Total elapsed (sum of 4 calls):          1,836.927 ms
    BLOCK0 Forward (kernel, model forward only): 459.232±1.888 ms/call
    Time per token:                              7.0073 us/token
    GPU (before run → after run):                57→61C, 855→870MHz, 28.52→67.55W
  [repeat 2/5]
    Tokens processed:                            262,144
    Total elapsed (sum of 4 calls):          1,860.481 ms
    BLOCK0 Forward (kernel, model forward only): 465.120±0.773 ms/call
    Time per token:                              7.0972 us/token
    GPU (before run → after

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 18629.04it/s]


# Clock test

In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation


## GPU warm up

In [ ]:
%cd "/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/first_layer_estimation/"
!python first_layer_estimation.py --dtype float32 --reversed_batch false --only_batch 1 --no_save

/content/drive/MyDrive/Project/nanoGPT/Batch Size vs Inference Speed/2_first_layer_estimation
loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
config.json: 100% 665/665 [00:00<00:00, 699kB/s]

model.safetensors: downloading bytes:  45% 246M/548M [00:01<00:01, 204MB/s, 21.4MB/s  ]
model.safetensors: downloading bytes:  79% 435M/548M [00:02<00:00, 163MB/s, 36.6MB/s  ]
model.safetensors: downloading bytes:  87% 474M/548M [00:03<00:00, 152MB/s, 38.7MB/s  ]
model.safetensors: reconstructing file:  73% 402M/548M [00:06<00:02, 50.3MB/s, 30.3MB/s  ]
model.safetensors: downloading bytes: 100% 474M/474M [00:06<00:00, 69.0MB/s, 40.0MB/s  ]
model.safetensors: reconstructing file: 100% 548M/548M [00:06<00:00, 79.7MB/s, 39.5MB/s  ]
Loading weights: 100% 148/148 [00:00<00:00, 4944.18it/s]
generation_config.json: 100% 124/124 [00:00<00:00, 531kB/s]
VAL_TOKENS=274,432 (max_batch=1, target_tokens=2

## Clock fixed to 585

In [ ]:
# !nvidia-smi -q -d SUPPORTED_CLOCKS | head -30
!nvidia-smi -lgc 585
!nvidia-smi --query-gpu=clocks.sm,clocks.max.sm,power.limit --format=csv

GPU clocks set to "(gpuClkMin 585, gpuClkMax 585)" for GPU 00000000:00:04.0

All done.
clocks.current.sm [MHz], clocks.max.sm [MHz], power.limit [W]
585 MHz, 1590 MHz, 70.00 W


In [ ]:
%%bash
mkdir -p output/power
mkdir -p output/ascending
rm -f output/ascending/fineweb_block0_nsight_clocklocked_585.csv

for B in 1 2 4 8 16 32 64; do
    nvidia-smi --query-gpu=timestamp,clocks.sm,power.draw,clocks_throttle_reasons.sw_power_cap,clocks_throttle_reasons.sw_thermal_slowdown \
        --format=csv -lms 200 > output/power/clocklock_585_block0_b${B}_log.csv &
    LOGGER_PID=$!

    python first_layer_estimation.py --dtype float32 --only_batch $B --csv_name fineweb_block0_nsight_clocklocked_585.csv

    kill $LOGGER_PID
done

nvidia-smi -rgc

loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
VAL_TOKENS=274,432 (max_batch=1, target_tokens=262,144)
Dtype: torch.float32
Batch sizes to sweep (ascending): [1]
Repeats per batch_size: 5

=== batch_size=1 (iters=256, tokens to process=262,144) ===
  [repeat 1/5]
    Tokens processed:                            262,144
    Total elapsed (sum of 256 calls):          3,482.269 ms
    BLOCK0 Forward (kernel, model forward only): 13.603±4.657 ms/call
    QKV_PROJ only (kernel, real/non-ncu):        1.922±1.979 ms/call
    Time per token:                              13.2838 us/token
    GPU (before run → after run):                40→42C, 585→585MHz, 26.45→53.12W
  [repeat 2/5]
    Tokens processed:                            262,144
    Total elapsed (sum of 256 calls):          2,541.470 ms
    BLOCK0 Forward (kernel, model forward only): 9.928±0.131 ms/call
    QKV_PROJ only (ker

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 14393.03it/s]


## Clock fixed to 900

In [ ]:
!nvidia-smi -lgc 900
!nvidia-smi --query-gpu=clocks.sm,clocks.max.sm,power.limit --format=csv

GPU clocks set to "(gpuClkMin 900, gpuClkMax 900)" for GPU 00000000:00:04.0

All done.
clocks.current.sm [MHz], clocks.max.sm [MHz], power.limit [W]
900 MHz, 1590 MHz, 70.00 W


In [ ]:
%%bash
mkdir -p output/power
mkdir -p output/ascending
rm -f output/ascending/fineweb_block0_nsight_clocklocked_900.csv

for B in 1 2 4 8 16 32 64; do
    nvidia-smi --query-gpu=timestamp,clocks.sm,power.draw,clocks_throttle_reasons.sw_power_cap,clocks_throttle_reasons.sw_thermal_slowdown \
        --format=csv -lms 200 > output/power/clocklock_900_block0_b${B}_log.csv &
    LOGGER_PID=$!

    python first_layer_estimation.py --dtype float32 --only_batch $B --csv_name fineweb_block0_nsight_clocklocked_900.csv

    kill $LOGGER_PID
done

nvidia-smi -rgc

loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
overriding dropout rate to 0.0
number of parameters: 123.65M
VAL_TOKENS=274,432 (max_batch=1, target_tokens=262,144)
Dtype: torch.float32
Batch sizes to sweep (ascending): [1]
Repeats per batch_size: 5

=== batch_size=1 (iters=256, tokens to process=262,144) ===
  [repeat 1/5]
    Tokens processed:                            262,144
    Total elapsed (sum of 256 calls):          1,944.047 ms
    BLOCK0 Forward (kernel, model forward only): 7.594±1.015 ms/call
    QKV_PROJ only (kernel, real/non-ncu):        1.050±0.024 ms/call
    Time per token:                              7.4160 us/token
    GPU (before run → after run):                55→57C, 900→900MHz, 28.51→62.24W
  [repeat 2/5]
    Tokens processed:                            262,144
    Total elapsed (sum of 256 calls):          1,975.860 ms
    BLOCK0 Forward (kernel, model forward only): 7.718±1.153 ms/call
    QKV_PROJ only (kerne

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 15195.27it/s]


In [ ]:
from google.colab import runtime
runtime.unassign()